# Agent 1: Poor Contact Expert (접촉불량 전문가)

In [1]:
# Poor Contact Expert Graph Init
import sys
import os
from pathlib import Path
import json
import time

# 프로젝트 루트 경로 설정
project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(project_root))

# 환경 변수 로드
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

print(f"✓ 프로젝트 루트: {project_root}")
# 2. 테스트 이미지 준비
from src.utils import find_data_directory
from src.tools.experts.expert_utils import save_bytes_to_temp_file

from src.prompts.contact_expert_prompts import get_final_verdict_prompt
from src.tools.experts.expert_utils import call_gemini_text, parse_json_response
try:
    data_dir = find_data_directory()
    # 테스트할 이미지 파일명 (필요시 변경)
    test_image_name = "Poor_Contact_001.jpg" 
    test_image_path = Path(data_dir) / test_image_name
    
    if not test_image_path.exists():
        image_files = list(Path(data_dir).glob("*.png")) + list(Path(data_dir).glob("*.jpg"))
        if image_files:
            test_image_path = image_files[0]
            print(f"⚠️ 지정된 이미지를 찾을 수 없어 {test_image_path.name}을 사용합니다.")
        else:
            raise FileNotFoundError("테스트할 이미지가 없습니다.")
            
    print(f"✓ 테스트 이미지: {test_image_path}")
    
    with open(test_image_path, 'rb') as f:
        image_data = f.read()
    temp_image_path = save_bytes_to_temp_file(image_data)
    print(f"✓ 임시 이미지 경로: {temp_image_path}")
    
except Exception as e:
    print(f"❌ 오류: {e}")
# 3. 그래프 빌드 및 상태 초기화 (실행 전 필수)
from src.graphs.contact_expert_graph import build_contact_expert_graph
from src.nodes.contact_nodes import ContactExpertState

print("✓ 그래프 빌드 및 초기 상태 생성 중...")

initial_state = ContactExpertState(
    messages=[],
    image_path=temp_image_path,
    hotspots=[],
    hotspot_queue=None,
    analysis_results=[],
    # Loop 변수 초기화
    current_hotspot=None,
    detector_result=None,
    roi_image_path=None,
    connection_type=None,
    terminal_result=None,
    splice_result=None,
    plug_result=None
)

graph = build_contact_expert_graph()
print("✓ 완료: initial_state 및 graph 객체 준비됨")

✓ 프로젝트 루트: c:\Users\user\Documents\Project\P_04_Scope
✓ 테스트 이미지: c:\Users\user\Documents\Project\P_04_Scope\data\Poor_Contact_001.jpg
✓ 임시 이미지 경로: C:\Users\user\AppData\Local\Temp\tmp_768_i6v.jpg


RealESRGANer or BasicSR not available (ImportError). Image enhancement will use simple resizing.


✓ 그래프 빌드 및 초기 상태 생성 중...
✓ 완료: initial_state 및 graph 객체 준비됨


In [2]:
# Poor Contact Expert Graph Run
print("=" * 60)
print("Multi-Hotspot Loop 전체 실행")
print("=" * 60)

start_time = time.time()
try:
    # Recursion Limit을 넉넉히 주어야 Loop가 돕니다.
    final_state = graph.invoke(initial_state, config={"recursion_limit": 50})
    elapsed = time.time() - start_time
    
    print(f"\n✓ 전체 실행 완료 (소요시간: {elapsed:.2f}초)")
    
    # 결과 출력
#     print("\n📊 최종 리포트 (Verdict):")
#     print("-" * 60)
#     print(final_state.get("verdict_report", "결과 없음"))
#     
    # Hotspots 확인
#     hotspots = final_state.get("hotspots", [])
#     print(f"\n🔍 발견된 Hotspots: {len(hotspots)}개")
#     for h in hotspots:
#         print(f"   - ID {h.get('id')}: {h.get('damage_type')} ({h.get('severity_score')}점)")
#     
    # 디버깅: final_state의 전체 키 확인
#     print(f"\n[Debug] Final State Keys: {list(final_state.keys())}")
# 
    # 분석된 결과들 내용 확인 (JSON 출력)
#     print("\n📄 [Detailed Analysis Results]")
#     print("-" * 60)
#     analysis_results = final_state.get("analysis_results", [])
#     print(json.dumps(analysis_results, indent=2, ensure_ascii=False))
# 
    # 결과 시각화 (Processed Images)
#     print("\n📷 [Processed Images Visualization]")
#     print("-" * 60)
#     
#     from PIL import Image
#     import matplotlib.pyplot as plt
#     import matplotlib.font_manager as fm
#     import os
#     
    # 한글 폰트 설정 (Windows)
#      try:
        # Windows에서 사용 가능한 한글 폰트 찾기
#          font_list = ['Malgun Gothic', 'NanumGothic', 'NanumBarunGothic', 'Gulim', 'Batang']
#          font_found = None
#          for font_name in font_list:
#              try:
#                  font_path = fm.findfont(fm.FontProperties(family=font_name))
#                  if font_path:
#                      plt.rcParams['font.family'] = font_name
#                      font_found = font_name
#                      break
#              except:
#                  continue
#          
#          if font_found:
#              print(f"✓ 한글 폰트 설정: {font_found}")
#          else:
            # 폰트를 찾지 못한 경우 경고만 출력하고 계속 진행
#              print("⚠️ 한글 폰트를 찾을 수 없습니다. 한글이 제대로 표시되지 않을 수 있습니다.")
#      except Exception as e:
#          print(f"⚠️ 폰트 설정 중 오류: {e}")
#      
#      if not analysis_results:
#          print("No analyzed images found.")
#      else:
#          num_images = len(analysis_results)
#          if num_images > 0:
            # 서브플롯 크기 및 배열 설정
#              cols = min(num_images, 5)
#              rows = (num_images - 1) // cols + 1
#              fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
#              
            # axes가 단일 객체이거나 1차원 배열일 경우 처리
#              if num_images == 1:
#                  axes = [axes]
#              elif rows > 1:
#                  axes = axes.flatten()
#              
            # 빈 서브플롯 숨기기
#              if hasattr(axes, '__len__') and len(axes) > num_images:
#                  for ax in axes[num_images:]:
#                      ax.axis('off')
#              
#              for idx, res in enumerate(analysis_results):
                # Data Extraction adapted for Partial Break structure
#                  h_info = res.get('hotspot_info', {})
#                  hotspot_id = h_info.get('id', '?')
#                  feature = h_info.get('damage_type', 'Unknown')
#                  
#                  roi_path = res.get('roi_image_path') # Fixed: use roi_image_path instead of roi_path
#                  spec_res = res.get('specialist_result', {}) or {}
#                  
#                  is_pb = spec_res.get('is_partial_break', False)
#                  conf = spec_res.get('confidence', 0)
#                  
#                  ax = axes[idx]
#                  
#                  if roi_path and os.path.exists(roi_path):
#                      try:
#                          img = Image.open(roi_path)
#                          ax.imshow(img)
                        # Title 구성: ID, 판정 결과, 신뢰도
#                          title_text = f"ID {hotspot_id}: {feature}\n"
#                          if is_pb:
#                              title_text += f"⚠️ Partial Break ({conf}%)"
#                          else:
#                              title_text += f"Normal / Other ({conf}%)"
#                              
#                          ax.set_title(title_text, fontsize=10, color='red' if is_pb else 'blue')
#                      except Exception as e:
#                          ax.text(0.5, 0.5, "Error Loading", ha='center')
#                          print(f"Error loading image {roi_path}: {e}")
#                  else:
#                      ax.text(0.5, 0.5, "Image Not Found", ha='center')
#                      ax.set_title(f"ID {hotspot_id}")
#                      
#                  ax.axis('off')
#              
#              plt.tight_layout()
#              plt.show()

except Exception as e:
    print(f"❌ 실행 중 오류: {e}")
    import traceback
    traceback.print_exc()

Multi-Hotspot Loop 전체 실행

📡 [Hotspot Detector] 다중 발화 지점 탐색 시작... (이미지: C:\Users\user\AppData\Local\Temp\tmp_768_i6v.jpg)


Upscale Error: RealESRGANer not initialized


📋 [Hotspot Detector] Finish reason: FinishReason.STOP

💭 [Hotspot Detector] 모델 응답 텍스트:
------------------------------------------------------------
📝 [Thinking 과정 (SDK Extracted)]:
**Analysis of Fire Scene Image: Electrical Components**

Alright, let's break this down systematically. My expertise demands absolute objectivity, so I'll stick strictly to observable characteristics using precise terminology. No jumping to conclusions here; only shape, color, location, size, and texture.

First, I'm presented with an image showing electrical wires and cables against a teal/green background. My initial assessment reveals a multi-core cable (labeled ②) and several single-core wires (labeled ①) in varying states of degradation. Red markers with numbers (3, 5) are also present.

The single-core wires on the left exhibit blackened insulation with exposed copper, displaying a charred texture. The bottom wire shows bare copper with reddish-brown oxidation and some green oxidation (verdigris) near 

Upscale Error: RealESRGANer not initialized


📝 [Result Aggregator] 결과 저장 (ID: 1)

▶️ [Hotspot Manager] Processing Hotspot ID 2 (Localized Copper Oxidation (국부적 구리 산화물 형성))
✂️ [ROI Crop] Hotspot 영역 크롭... [165, 385, 275, 465]
✨ [Enhancement] ROI 이미지 2배 향상 적용 중...
✨ [Enhancement] 향상 완료: C:\Users\user\AppData\Local\Temp\roi_crop_we31t63l.jpg

🔍 [Component Classifier] 부품 유형 식별 중... (Dual Input: Context + Detail, ROI: C:\Users\user\AppData\Local\Temp\roi_crop_we31t63l.jpg)
👁️ [Observation] 여러 가닥의 구리 전선이 한데 모여 있으며, 그 끝부분에 녹색의 뭉툭한 원뿔형 캡(Cap) 형태의 물체가 체결되어 있음. 주변 전선 피복은 열에 의해 변색되거나 소실되었으나, 결선 부위를 감싸는 캡의 외형은 비교적 뚜렷하게 남아 있음.
✅ 판별 결과: Splice (신뢰도: 95%)

🔗 [Splice Specialist] 정밀 분석 수행...
📝 [Result Aggregator] 결과 저장 (ID: 2)

🏁 [Hotspot Manager] 모든 Hotspot 처리 완료.
📋 [Contact Verdict] Finish reason: FinishReason.STOP

💭 [Contact Verdict] 모델 응답 텍스트:
------------------------------------------------------------
📝 [Thinking 과정 (SDK Extracted)]:
**Analysis of Fire Cause: Contact Failure (High Probability)**

Alright, here's how I'm approaching this, as